In [1]:
from plotly.io import show
from sklearn.model_selection import train_test_split

from skfolio import Population, RiskMeasure
from skfolio.datasets import load_sp500_dataset
from skfolio.optimization import InverseVolatility, RiskBudgeting
from skfolio.preprocessing import prices_to_returns

prices = load_sp500_dataset()

X = prices_to_returns(prices)
X_train, X_test = train_test_split(X, test_size=0.33, shuffle=False)

In [2]:
risk_budget = {asset_name: 1 for asset_name in X.columns}
risk_budget["AAPL"] = 1.5
risk_budget["GE"] = 0.2
risk_budget["JPM"] = 0.2

In [3]:
model = RiskBudgeting(
    risk_measure=RiskMeasure.CVAR,
    risk_budget=risk_budget,
    portfolio_params=dict(name="Risk Budgeting - CVaR"),
)
model.fit(X_train)
model.weights_

array([0.06173004, 0.03257481, 0.03336054, 0.0399478 , 0.06349021,
       0.00930188, 0.04579208, 0.06880871, 0.00765287, 0.06700294,
       0.05537205, 0.05430755, 0.04828151, 0.0703356 , 0.05371263,
       0.06892867, 0.04230958, 0.04897346, 0.0625168 , 0.06560027])

In [4]:
bench = InverseVolatility(portfolio_params=dict(name="Inverse Vol"))
bench.fit(X_train)
bench.weights_

array([0.03306735, 0.02548697, 0.03551377, 0.0296872 , 0.06358463,
       0.05434705, 0.04742354, 0.07049715, 0.03882539, 0.06697905,
       0.05570808, 0.05576851, 0.04723274, 0.06351213, 0.05581397,
       0.0676481 , 0.02564642, 0.03970752, 0.05744543, 0.06610498])

In [5]:
ptf_model_train = model.predict(X_train)
fig = ptf_model_train.plot_contribution(measure=RiskMeasure.CVAR)
show(fig)

In [6]:
ptf_bench_train = bench.predict(X_train)
ptf_bench_train.plot_contribution(measure=RiskMeasure.CVAR)

In [7]:
ptf_model_test = model.predict(X_test)
ptf_bench_test = bench.predict(X_test)

In [8]:
population = Population([ptf_model_test, ptf_bench_test])

In [9]:
population.plot_composition()

In [10]:
fig = population.plot_cumulative_returns()
show(fig)

In [11]:
population.summary()

,Risk Budgeting - CVaR,Inverse Vol
Mean,0.067%,0.064%
Annualized Mean,17.00%,16.06%
Variance,0.010%,0.010%
Annualized Variance,2.57%,2.56%
Semi-Variance,0.0052%,0.0053%
Annualized Semi-Variance,1.32%,1.33%
Standard Deviation,1.01%,1.01%
Annualized Standard Deviation,16.02%,16.00%
Semi-Deviation,0.72%,0.73%
Annualized Semi-Deviation,11.50%,11.54%
